# Lesson 02: Front door, back door

> **Question.** What rent must the property earn to support its cost? What land price
> can the reference rents support?
>
> **You will learn to:**
>
> 1. make a Model, edit it in a draft and commit it, with string equations;
> 2. tell an equation from a calculation direction;
> 3. fix values in a Specification, solve, and read the result from `Outcome`;
> 4. reverse the question by changing two roles;
> 5. (optional) recognise underdetermined, infeasible and unsupported outcomes.
>
> **Before you start.** Lesson 01.
>
> **Time.** 60 minutes core, 20 minutes optional.
>
> **Not in this lesson.** Dates and cash flows over time (lesson 03). The project
> graph (lesson 06): here the property is one asset, with totals by use.
>
> **Source.** Simple financial feasibility analysis (SFFA), Geltner et al. (2007),
> section 28.4.1. The method is cited; the exhibits are not reproduced. The numbers
> are the property's own.

## 1. The property as one asset

The mixed-use property has two buildings over a shared basement. In this lesson we
use only its totals in a stabilised year, 2032: the lettable area, reference rent and
vacancy of each use, the operating cost and the works cost. The data package records
the origin of each number.

In [ ]:
import helpers

import rangekeeper as rk

In [ ]:
data = helpers.load_property(year=2032)
data  # key, quantity and origin of each total

This is the property. Its totals come from this design. Lesson 06 opens the
design and shows the objects behind each total.

In [ ]:
helpers.show_geometry(name="mixed_use_property")

## 2. A calculation has a direction

A spreadsheet calculates in one direction. Start from the cost and calculate the
income that a lender needs. Do it once in plain Python. A Quantity's `magnitude` is a
`Decimal`, so `float()` makes it a plain number.

Then ask the reverse question: what land price does a given income support? The
spreadsheet needs new formulas. The equations did not change; only the question did.

In [ ]:
total_cost = 20e6 + float(data["works_cost"].magnitude)
debt_service = 0.0860 * 0.65 * total_cost
required_noi = 1.25 * debt_service
required_noi  # AUD per year, calculated from cost to income

## 3. Declare the equations once

A Model describes things, called entities, with their values and the equations
between them. Here it describes one entity: the project, the whole property as one
asset. An equation does not say which side is known.

A Model never changes. To change one, `edit()` gives a draft. The draft collects the
changes; `commit()` checks them all once, including the units of every equation, and
returns a new Model. The first Model is empty; the draft adds the project, its values
and its equations.

In [ ]:
model = rk.Model(name="Mixed-use feasibility")
draft = model.edit()
project = draft.add_entity(code="project", classification="project")
project.value(key="land_cost", units="AUD")
project.value(key="works_cost", units="AUD")
project.value(key="total_cost", units="AUD")
project.value(key="leverage", units="ratio")
project.value(key="mortgage", units="AUD")
project.value(key="mortgage_constant", units="% pa")
project.value(key="debt_service", units="AUD pa")

The cost and loan values are declared. Now the income values.

In [ ]:
project.value(key="coverage", units="ratio")
project.value(key="noi", units="AUD pa")
project.value(key="operating_cost", units="AUD pa")
project.value(key="effective_income", units="AUD pa")
project.value(key="rent_factor", units="ratio")
project.value(key="reference_income", units="AUD pa")

In [ ]:
project.equations(
    "total_cost = land_cost + works_cost",
    "mortgage = leverage * total_cost",
    "debt_service = mortgage_constant * mortgage",
    "noi = coverage * debt_service",
    "effective_income = noi + operating_cost",
    "effective_income = rent_factor * reference_income",
)

The reference income keeps the different rents and vacancies of each use. One
dimensionless rent factor scales all of them: 1 means the reference rents, 1.1 means
10 % higher. Parking earns no rent.

In [ ]:
project.value(key="hotel_area", units="sqm")
project.value(key="hotel_rent", units="AUD per sqm pa")
project.value(key="hotel_vacancy", units="ratio")
project.value(key="residential_area", units="sqm")
project.value(key="residential_rent", units="AUD per sqm pa")
project.value(key="residential_vacancy", units="ratio")
project.value(key="retail_area", units="sqm")
project.value(key="retail_rent", units="AUD per sqm pa")
project.value(key="retail_vacancy", units="ratio")

In [ ]:
project.equations(
    "reference_income = hotel_area * hotel_rent * (1 - hotel_vacancy)"
    " + residential_area * residential_rent * (1 - residential_vacancy)"
    " + retail_area * retail_rent * (1 - retail_vacancy)"
)
model = draft.commit()

In [ ]:
helpers.show_equations(of=model)  # a diagram with no input or output arrows

> **Under the hood: the same equation in Python.** A later release adds
> `rangekeeper.expressions`, imported as `rx`, which writes equations with Python
> operators ([#309](https://github.com/daniel-fink/rangekeeper/issues/309)). A
> declaration will return a handle, and `project.ref(key=)` will find the handle of a
> value declared earlier, so `noi == coverage * debt_service` makes the same records
> as the text `"noi = coverage * debt_service"`. Both forms use the same names.

## 4. The mortgage constant, by hand once

The mortgage constant is the annual debt service per dollar of loan. It is not the
interest rate. With monthly payments over 20 years at 6 % a year:

In [ ]:
r, n = 0.06 / 12, 20 * 12
12 * r / (1 - (1 + r) ** -n)  # 0.0860 per year

The library has one general loan calculation: the payment, like a spreadsheet's
PMT, but with units. The mortgage constant is the payment on one dollar of loan, per
year. Check that it agrees with the calculation by hand.

In [ ]:
monthly = rk.duration.Frequency.MONTH
payment = rk.finance.payment(
    principal="1 AUD", rate="6 % pa", term="20 years", frequency=monthly
)
payment  # the monthly payment on each dollar of loan

In [ ]:
constant = (payment / rk.Quantity(text="1 AUD")).to(units="% pa")
constant  # 8.60 % pa

> **Finance note.** Loan-to-value (leverage) limits the loan to a share of value.
> SFFA uses total development cost as a proxy for value. The debt-service coverage
> ratio requires NOI to exceed debt service by a margin.

## 5. The front door

**Predict:** at a land cost of 20 million AUD, must rents be above or below the
reference rents?

A Specification asks the question. It fixes the known values, by name. Every value
that it does not fix is free: the solve finds it. Each `fix` returns a new
Specification, so the fixes can be grouped by topic.

In [ ]:
front = (
    model.specify(name="Front door")
    .fix(land_cost="20e6 AUD", works_cost=data["works_cost"])
    .fix(operating_cost=data["operating_cost"])
    .fix(leverage="65 %", coverage=1.25, mortgage_constant=constant)
)

The reference rents and areas by use, from the data package. Every fixed value is
named.

In [ ]:
front = front.fix(
    hotel_area=data["hotel_area"],
    hotel_rent=data["hotel_rent"],
    hotel_vacancy=data["hotel_vacancy"],
    residential_area=data["residential_area"],
    residential_rent=data["residential_rent"],
    residential_vacancy=data["residential_vacancy"],
    retail_area=data["retail_area"],
    retail_rent=data["retail_rent"],
    retail_vacancy=data["retail_vacancy"],
)
front.roles  # each value: fixed with its quantity, or free for the solve to find

In [ ]:
front_outcome = rk.execution.solve(specification=front)
front_outcome.raise_for_status()
front_outcome["rent_factor"]

The rent factor applies to every use. Show the required rent beside the reference
rent. A required rent is a threshold, not a forecast.

In [ ]:
factor = front_outcome["rent_factor"]
required_rent = {
    "hotel": factor * data["hotel_rent"],
    "residential": factor * data["residential_rent"],
    "retail": factor * data["retail_rent"],
}
required_rent

## 6. The back door

**Predict:** at the reference rents, can the property support a land price of 20
million AUD?

Change two roles: fix the rent factor at 1 and free the land cost. The Model does not
change.

In [ ]:
back = front.fix(rent_factor=1).free("land_cost")
back_outcome = rk.execution.solve(specification=back)
back_outcome["land_cost"]

In [ ]:
helpers.show_equations(of=back)  # the same equations, with different markers

In [ ]:
rk.execution.compare(
    outcomes={"Front door": front_outcome, "Back door": back_outcome},
    keys=["land_cost", "rent_factor", "noi", "mortgage"],
)

> **Finance note.** A negative supportable land price is a shortfall under these
> assumptions. It is not a price that anyone can pay. The back-door number is a cost
> threshold, not a valuation of the land.

## 7. Round trip

Put the front door's rent factor into the back door. The solve must give back the
land cost that we started with.

In [ ]:
round_trip = back.fix(rent_factor=front_outcome["rent_factor"])
rk.execution.solve(specification=round_trip)["land_cost"]  # 20e6 AUD

## 8. Change one input

**Predict:** if the lender allows only 50 % leverage, does the supportable land price
rise or fall? Change one fixed value.

In [ ]:
lower_leverage = back.fix(leverage="50 %")
lower_leverage_outcome = rk.execution.solve(specification=lower_leverage)
lower_leverage_outcome["land_cost"] - back_outcome["land_cost"]

It rises. The income fixes the debt service, so it fixes the loan. With less
leverage, the same loan is a smaller share of a larger total cost. SFFA asks for no
return on the equity that pays the rest, so the supportable cost rises.

## 9. Optional: when the question is not well posed

A solve can fail in three different ways. Each failure has its own status and
diagnostics.

**Too few fixed values.** Free both the land cost and the rent factor. Many pairs fit,
and each pair changes every value between them.

In [ ]:
loose = back.free("rent_factor")
loose_outcome = rk.execution.solve(specification=loose)
(
    loose_outcome.status,
    loose_outcome.free_unknowns,
)  # UNDERDETERMINED, ('land_cost', 'total_cost', 'mortgage', 'debt_service', 'noi',
# 'effective_income', 'rent_factor')

**Too many fixed values that disagree.** Fix both the land cost and the rent
factor.

In [ ]:
tight = front.fix(rent_factor=1)
tight_outcome = rk.execution.solve(specification=tight)
(
    tight_outcome.status,
    tight_outcome.diagnostics,
)  # INFEASIBLE; the diagnostics give the solver's evidence

**A product of two unknowns.** Ask for leverage and coverage together.

In [ ]:
both_free = back.fix(land_cost="20e6 AUD").free("leverage", "coverage")
both_free_outcome = rk.execution.solve(specification=both_free)
both_free_outcome.status, both_free_outcome.diagnostics
# UNSUPPORTED: nonaffine_unknowns in "noi = coverage * debt_service"

In [ ]:
try:
    both_free_outcome.raise_for_status()
except rk.errors.UnsupportedError as error:
    print(error.code, error.equation)

## Exercises

**Try it 1: the largest loan.** At a land cost of 20 million AUD and reference rents,
what is the highest leverage the income supports?

In [ ]:
max_leverage = front.fix(rent_factor=1).free("leverage")
rk.execution.solve(specification=max_leverage)["leverage"]

**Try it 2: a longer amortisation.** With a 25-year amortisation, what rent factor
does the front door need?

In [ ]:
longer = rk.finance.payment(
    principal="1 AUD", rate="6 % pa", term="25 years", frequency=monthly
)
longer_constant = (longer / rk.Quantity(text="1 AUD")).to(units="% pa")
longer_outcome = rk.execution.solve(
    specification=front.fix(mortgage_constant=longer_constant)
)
assert longer_outcome["rent_factor"] < factor

**Try it 3: a stronger market.** If rents were 10 % above the reference rents,
what land price could they support? Predict the change before you solve.

In [ ]:
stronger = back.fix(rent_factor=1.1)
stronger_outcome = rk.execution.solve(specification=stronger)
assert stronger_outcome["land_cost"] > back_outcome["land_cost"]

## Checks and recap

The checks confirm the lesson's results.

In [ ]:
assert rk.quantity.isclose(
    actual=rk.execution.solve(specification=round_trip)["land_cost"],
    expected=rk.Quantity(text="20e6 AUD"),
)
assert rk.quantity.isclose(
    actual=front_outcome["effective_income"],
    expected=factor * front_outcome["reference_income"],
)
assert back.model_id == front.model_id == model.id

**Recap.**

1. A Model declares its equations once. An equation has no calculation direction.
2. A Specification asks the question: fix what you know, and the solve finds the
   rest. The back door changes two roles, not the Model.
3. A solve can be underdetermined, infeasible or unsupported. Each status names its
   cause.

**Next.** Lesson 03 puts the property on a timeline, compares named market cases and
adds debt, equity and funding.